In [ ]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWFjaGEvRHJvcGJveC8wNF90ZWFjaGluZy9BUjEwNDk4X0VNYUMxL1Byb2dQeVF1YXJ0by9Ob3RlYm9va3M='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


xvals = np.array([1,2,3,4])
yvals = xvals**2

plt.plot(xvals, yvals)
plt.xlabel('x')
plt.ylabel('$x^2$')

In [ ]:
fig, ax = plt.subplots()

In [ ]:
pi=np.pi

# create a single set of axes on a figure
fig, ax = plt.subplots()

xvals = np.linspace(-pi,pi,100)
sinx = np.sin(xvals)

ax.plot(xvals, sinx)

In [ ]:
fig, axs = plt.subplots(2)

axs[0].plot(xvals, np.sin(xvals))
axs[1].plot(xvals, np.cos(xvals))

In [ ]:
fig, axs = plt.subplots(2, 2)

[[ax1, ax2], [ax3, ax4]] = axs

ax1.plot(xvals, np.sin(xvals)) # x vs sin(x)
ax2.plot(np.sin(xvals), xvals) # flipped axes
ax3.plot(xvals, np.cos(xvals)) # x vs cos(x)
ax4.plot(np.sin(xvals), np.cos(xvals)) # parametric plot of sin(x) vs cos(x)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xvals = np.arange(11)
yvals = xvals**2

# set some x and y dimensions (in inch !!!)
dims = (8,8)
fig, ax = plt.subplots(figsize=dims) # specify the size of the figure

ax.plot(xvals, yvals, linestyle='--', marker='o', color='red')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
pi = np.pi

xs = np.linspace(0, 2*pi, 50)
ys = np.sin(xs)
zs = np.cos(xs)

fig, ax = plt.subplots()

ax.plot(xs, ys)
ax.plot(xs, zs)

In [ ]:
ax.set_xlabel("Angle t (radians)", size=12)
ax.set_ylabel("f (t)", size=12)

labels=["sin(t)", "cos(t)"]
ax.legend(labels)

fig # shows the figure again

In [ ]:
# run the code above (or paste it here) before running this cell

plt.savefig('sinusoids.png')

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def plotter(xdata, ydata):
    fig,ax = plt.subplots(1,2, figsize=(10,4))
    ax[0].plot(xdata,ydata, linewidth=0.2, color='red', marker='*', markerfacecolor='green')
    ax[1].hist(ydata, orientation='horizontal', edgecolor='black')
    ax[0].set_ylabel('Data Values', size=12)
    ax[0].set_xlabel('Data Index', size=12)
    ax[1].set_xlabel('Data Counts', size=12)
    ax[0].set_xlim(0,300)
    for i in [0,1]:
        ax[i].grid() 
        ax[i].set_ylim(0,80)
    ax[0].text(-50,0,'(a)', size=15)
    ax[1].text(-10,0,'(b)', size=15)
    
    return fig

##########
x = np.arange(1,301)
y = np.random.normal(42,12,300)
fig = plotter(x,y)
##########

fig.savefig('fancyfig.pdf')

In [ ]:
import numpy as np

thedata = dict() #another way of making an empty dictionary

thedata["Days"] = list(range(1,8))
thedata["Temperatures"] = np.array([23.3, 16.9, 21.8, 26.9, 23.5, 21.7, 19.8])

thedata

In [ ]:
thedata["Temperatures"] = thedata["Temperatures"]-10

thedata["Temperatures"]

In [ ]:
import pandas as pd

# make a dataframe called df out of our previous dictionary:
df = pd.DataFrame(thedata)
print(df)

In [ ]:
df

In [ ]:
df["Temperatures"]

In [ ]:
average = df["Temperatures"].mean()
print(average)

deviation = df["Temperatures"]-average
print(deviation)

In [ ]:
df["Deviations"] = deviation

df

In [ ]:
df.to_excel("devdata.xlsx", index=False, sheet_name="Temperature Deviations")

In [ ]:
df.to_csv("devdata.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd

temps = np.array([23.3, 16.9, 21.8, 26.9, 23.5, 21.7, 19.8])
average = temps.mean()
devs = temps-average # deviation of each point from the mean
stdev = temps.std() # standard deviation of the dataset
zscore = devs/stdev # Z-score is the number of standard deviations each reading is from the mean
data = [temps, devs, zscore]

mynewdataframe = pd.DataFrame(data)
mynewdataframe

In [ ]:
# rotate the dataframe:
newdata = mynewdataframe.transpose()
newdata

In [ ]:
newdata.columns = "Temp", "Deviation", "Z-value"

newdata

In [ ]:
newdata.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

newdata

In [ ]:
import pandas as pd

data = pd.read_excel("Files/Case Study Data.xlsx", sheet_name="Data")

data

In [ ]:
projects = data["Project"]

print(projects)

In [ ]:
selected_columns = ["Total.Area.sqm", "Cost.Million.USD"]

data2 = data[selected_columns]

data2.index = projects # from the previous cell

data2

In [ ]:
import pandas as pd

data = pd.read_excel("Files/Case Study Data.xlsx", sheet_name="Data")

cost = data["Cost.Million.USD"]
area = data["Total.Area.sqm"]
costpermetre2 = cost/area

# create a new column
data["Cost.Per.sqm"] = costpermetre2


selected_columns = ["Project", "Total.Area.sqm", "Cost.Million.USD", "Cost.Per.sqm"]

# create a new DataFrame with the selected columns as the data and set the project name column as the index
newdata = data[selected_columns].set_index("Project")

newdata

In [ ]:
ax = newdata["Total.Area.sqm"].plot(kind="bar", figsize=(15,5)) # returns a set of figure axes

In [ ]:
ax1 = newdata["Total.Area.sqm"].plot.hist(edgecolor="black")
ax2 = newdata.plot.scatter(x="Total.Area.sqm", y="Cost.Million.USD")

In [ ]:
ax2.figure.savefig("scatterplot.png") # how to access the figure canvas and save it

In [ ]:
sortdata = newdata.sort_values(by="Total.Area.sqm")

ax3 = sortdata.plot.bar(subplots=True, figsize=(15,9))

In [ ]:
costmilUSD = data2["Cost.Million.USD"] #select the column 

condition = costmilUSD>100 #filter selected values

data3 = data2[condition]

data3